In [1]:
%pip install pandas
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [47]:
import pandas as pd 
import numpy as np
import re

In [87]:
data = pd.read_excel(r'C:\Users\KS\Desktop\สต็อกสินค้า1.xlsx' ,engine='calamine',header=11,usecols="A:F")
data.rename(columns={'Unnamed: 0':'DATE','Unnamed: 1':'Bill','เพิ่ม ':'details','Unnamed: 3':'import','ลด ':'export','คงเหลือ ':'balance'} ,inplace=True)

In [88]:
data['product_id'] = data.loc[data['DATE'] =='รหัสสินค้า', 'export']
data['product_id'].ffill(inplace=True)

In [89]:
data['unit'] = data.loc[data['DATE'].astype(str).str.strip() == 'คลัง', 'balance']
data['unit'].ffill(inplace=True)
data['unit'] = data['unit'].str.extract(r'(\d+)').fillna(0).astype(int)

In [90]:
# แปลงตรงๆ โดยบอกสไตล์ปฏิทินสากลไปก่อน
data['DATE'] = pd.to_datetime(data['DATE'], format='%d/%m/%Y', errors='coerce')

# ลบปีออก 543 ปี (ใช้ DateOffset)
data['DATE'] = data['DATE'] - pd.DateOffset(years=543)

In [92]:
datas

NameError: name 'datas' is not defined

 Robust Z-score

In [ ]:
%pip install scipy

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement scipy (from versions: none)

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for scipy


In [ ]:
%pip install python-calamine
%pip install pandas
%pip install openpyxl

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [8]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import openpyxl

# ==========================================
# 1. โหลดข้อมูล (ใส่ engine='calamine' เพื่อเลี่ยง XML เสียหายจากรอบแรก)
# ==========================================
file_path = r"C:\Users\KS\Desktop\ปวดหลัง.xlsx"
df = pd.read_excel(file_path, engine="calamine")

# ล้างช่องว่างที่อาจมองไม่เห็นในชื่อคอลัมน์ทิ้งให้หมดเพื่อความปลอดภัย
df.columns = df.columns.str.strip()

# [เสริมเกราะ 1] แปลง ID ให้เป็น string ทั้งหมด ป้องกันกรณี Excel แปลงบางตัวเป็นตัวเลขแล้วกลุ่มเพี้ยน
df['product_id'] = df['product_id'].astype(str).str.strip()

# เอาเฉพาะยอดนำเข้าที่มากกว่า 0 เท่านั้น (ตัด Noise/บิลยกเลิก ออก)
df = df[df['import'] > 0]

# ==========================================
# 2. คำนวณหา Outlier 
# ==========================================
# หา IQR และคูณสเกล 1.4826 สำหรับ แผน A
q1 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.25))
q3 = df.groupby('product_id')['import'].transform(lambda x: x.quantile(0.75))
iqr_scaled = (q3 - q1) * 1.4826

# หา Median และจำนวนบิล
group_median = df.groupby("product_id")["import"].transform("median")
group_count = df.groupby('product_id')['import'].transform('count')
group_sd = df.groupby("product_id")["import"].transform("std")
# ✨ [แก้ไขจุดที่ 1] หา MAD ดิบจาก SciPy แล้วค่อยคูณสเกล 1.4826 ข้างนอก 
mad_raw = df.groupby("product_id")["import"].transform(stats.median_abs_deviation)
group_mad_scaled = mad_raw * 1.4826

# เงื่อนไขที่ 2 ไม่ให้สูงเกิน 30% ของค่ากลาง
max_allowed_deviation = np.maximum(group_median * 0.3, 1.0)
group_mad_scaled = np.minimum(group_mad_scaled, max_allowed_deviation)
group_mad_scaled = np.maximum(group_mad_scaled, 1.0) # กันตัวหารเป็น 0

# เงื่อนไขการแบ่งกลุ่มสินค้า
conditions = [
    (iqr_scaled > 0) & (group_count >= 10),   # แผน A
    (iqr_scaled == 0) | (group_count < 10)    # แผน B
]

# คำนวณคะแนนดิบ Z-score
df["Adaptive_ZScore"] = np.select(conditions, [
    ((df["import"] - group_median) / iqr_scaled),       # แผน A 
    ((df["import"] - group_median) / group_mad_scaled)  # แผน B 
], default=0)

# min max mean 
df['min'] = df.groupby('product_id')['import'].transform('min')
df['max'] = df.groupby('product_id')['import'].transform('max')
df['mean'] = df.groupby('product_id')['import'].transform('mean')
df["z_score"] = (df["import"] - df['mean']) / group_sd
df["median"] = group_median
df["IQR"] = (q3 - q1)


# ==============================================================================
# 📊 สเกล Z-Score / SD แปลงเป็นเปอร์เซ็นต์ข้อมูลที่ครอบคลุม (Normal Distribution)
# ==============================================================================
#  ± 1 SD  = ครอบคลุมข้อมูล  68.27%  (โอกาสหลุดเกณฑ์ ~ 31.73%)
#  ± 2 SD  = ครอบคลุมข้อมูล  95.45%  (โอกาสหลุดเกณฑ์ ~  4.55%)
#  ± 3 SD  = ครอบคลุมข้อมูล  99.73%  (โอกาสหลุดเกณฑ์ ~  0.27% -> Outlier)
# ==============================================================================
df["is_outlier"] = df['Adaptive_ZScore'].abs() > 3

# ==========================================
# 3. เจาะลึกระดับบิล (Drill-Down)
# ==========================================
# ดึงรายชื่อรหัสสินค้าทั้งหมดที่มีแถวใดแถวหนึ่งติดสถานะ Outlier
final_report = df[df["is_outlier"] == True].copy()

# ✨ [แก้ไขจุดที่ 2] จัดเรียงข้อมูลพร้อมใส่วงเล็บปิดให้สมบูรณ์
final_report = final_report.sort_values(
    by=["product_id", "Adaptive_ZScore"], ascending=[True, False]
)

In [26]:
#final_report.to_excel(r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')

In [3]:
final_report[['DATE','Bill','product_id','import','min','max','mean','median','IQR','z_score','Adaptive_ZScore']].reset_index(drop=True
                                                                                    )

,DATE,Bill,product_id,import,min,max,mean,median,IQR,z_score,Adaptive_ZScore
0,2026-03-14,IBK3256903/039,กะซ้ง น้ำดื่ม 850 มล.,600.00,1.000,600.00,113.161765,120.0000,60.000,5.169963,5.395926
1,01/03/2569,.,กาแฟซุปเปอร์มิก แดง (หีบ30ซอง*30),8.07,0.010,8.07,1.865946,2.0000,1.000,4.364932,4.094159
2,05/06/2569,IBK3256906/012,กาแฟซุปเปอร์มิก แดง (หีบ30ซอง*30),8.00,0.010,8.07,1.865946,2.0000,1.000,4.315683,4.046945
3,12/07/2569,IBK3256907/031,กาแฟซุปเปอร์มิก แดง (หีบ30ซอง*30),7.00,0.010,8.07,1.865946,2.0000,1.000,3.612122,3.372454
4,27/03/2569,IBK3256903/072,กาแฟโกลเด้น หีบ(30ซอง*20),40.00,1.000,40.00,2.772727,1.0000,1.000,6.065610,26.305140
5,14/03/2569,IBK3256903/039,กาแฟโกลเด้น หีบ(30ซอง*20),15.00,1.000,40.00,2.772727,1.0000,1.000,1.992246,9.442871
6,06/05/2569,IBK3256905/031,คินเดอร์ ทรอนกี้ 18ก.,24.00,8.000,24.00,14.666667,12.0000,8.000,1.120897,3.333333
7,13/04/2569,IBK3256904/044,น้ำ เพียวไลฟ์330,248.00,1.000,248.00,31.106667,14.0000,14.000,4.027147,11.273631
8,13/05/2569,IBK3256905/039,น้ำ เพียวไลฟ์330,248.00,1.000,248.00,31.106667,14.0000,14.000,4.027147,11.273631
9,04/06/2569,IBK3256906/009,น้ำ เพียวไลฟ์330,248.00,1.000,248.00,31.106667,14.0000,14.000,4.027147,11.273631


In [4]:
final_report[['DATE','Bill','product_id','import','min','max','mean','median','IQR','z_score','Adaptive_ZScore']].reset_index(drop=True).to_excel(
    r'C:\Users\KS\Desktop\product_out1.xlsx',index=False,engine='openpyxl')
                                                                                    